# 😎 Playground 4: StyleGAN Latent Studio – Đại Số Vector Mặt Người & Biến Hình
### Khám phá Không gian Tiềm ẩn: Phép cộng trừ thuộc tính khuôn mặt và biến hình mượt mà (Morphing)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/04_stylegan_latent_studio.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. Hiểu **Không gian tiềm ẩn (Latent Space $\mathcal{Z}$ hoặc $\mathcal{W}$)**: Nơi mỗi vector số thực đại diện cho một khuôn mặt hoặc ý tưởng độc đáo.
2. Thực hiện **Đại số Vector (Vector Arithmetic)**:
   $$\vec{z}_{mặt\_cười\_có\_kính} = \vec{z}_{gốc} + \alpha \cdot \vec{v}_{cười} + \beta \cdot \vec{v}_{kính\_râm}$$
3. Thực hiện **Nội suy tuyến tính (Linear Interpolation / Latent Morphing)** để xem khuôn mặt Người A biến đổi mượt mà sang Người B.

### 1. Cài đặt môi trường & Kiểm tra thiết bị

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {device}")

### 2. Xây dựng Deep Convolutional Generator (DCGAN Face Generator 32x32)

In [ ]:
class FaceGenerator(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.conv = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 8x8
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 4, 4)
        return self.conv(x)

latent_dim = 32
G_face = FaceGenerator(latent_dim=latent_dim).to(device)
G_face.eval()
print("✓ Khởi tạo FaceGenerator thành công!")

### 3. Thử nghiệm Đại Số Vector: Thêm Nụ Cười 😄 và Kính Râm 🕶️

In [ ]:
torch.manual_seed(42)
# 1. Vector gốc đại diện cho khuôn mặt bình thường
z_base = torch.randn(1, latent_dim, device=device)

# 2. Vector hướng thuộc tính (Attribute Directions)
v_smile = torch.randn(1, latent_dim, device=device) * 0.4
v_glasses = torch.randn(1, latent_dim, device=device) * 0.5

# Phép toán đại số
z_smile = z_base + 1.2 * v_smile
z_glasses = z_base + 1.5 * v_glasses
z_combo = z_base + 1.2 * v_smile + 1.5 * v_glasses

with torch.no_grad():
    img_base = (G_face(z_base)[0].permute(1, 2, 0).cpu() + 1) / 2
    img_smile = (G_face(z_smile)[0].permute(1, 2, 0).cpu() + 1) / 2
    img_glasses = (G_face(z_glasses)[0].permute(1, 2, 0).cpu() + 1) / 2
    img_combo = (G_face(z_combo)[0].permute(1, 2, 0).cpu() + 1) / 2

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
axes[0].imshow(img_base.clamp(0, 1)); axes[0].set_title("1. Mặt Gốc $z_{base}$"); axes[0].axis('off')
axes[1].imshow(img_smile.clamp(0, 1)); axes[1].set_title("2. + Vector Nụ Cười"); axes[1].axis('off')
axes[2].imshow(img_glasses.clamp(0, 1)); axes[2].set_title("3. + Vector Kính Râm"); axes[2].axis('off')
axes[3].imshow(img_combo.clamp(0, 1)); axes[3].set_title("4. Vừa Cười Vừa Đeo Kính"); axes[3].axis('off')
plt.suptitle("Phép Toán Đại Số Vector Trong Không Gian Tiềm Ẩn (Vector Arithmetic)", fontsize=14, fontweight='bold')
plt.show()

### 4. Phép Biến Hình Mượt Mà Giữa 2 Người (Latent Morphing Interpolation)

$$\vec{z}_{interp}(\alpha) = (1 - \alpha) \cdot \vec{z}_A + \alpha \cdot \vec{z}_B \quad (\text{với } \alpha \in [0, 1])$$

In [ ]:
z_person_A = torch.randn(1, latent_dim, device=device)
z_person_B = torch.randn(1, latent_dim, device=device)

num_steps = 7
alphas = np.linspace(0, 1, num_steps)
morph_steps = []

with torch.no_grad():
    for a in alphas:
        z_step = (1 - a) * z_person_A + a * z_person_B
        img_step = (G_face(z_step)[0].permute(1, 2, 0).cpu() + 1) / 2
        morph_steps.append(img_step.clamp(0, 1).numpy())

fig, axes = plt.subplots(1, num_steps, figsize=(15, 3))
for i, img in enumerate(morph_steps):
    axes[i].imshow(img)
    axes[i].set_title(f"{int(alphas[i]*100)}% (A ➔ B)")
    axes[i].axis('off')

plt.suptitle("Quá Trình Biến Đổi Mượt Mà (Latent Space Walk)", fontsize=14, fontweight='bold')
plt.show()